In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin


In [3]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (891, 12)
Test shape: (418, 11)


In [4]:
print(train.head())
print(train.info())
print(train.describe(include="all"))

print("\nMissing values:")
print(train.isnull().sum())

print("\nTarget distribution:")
print(train["Survived"].value_counts())
print(train["Survived"].value_counts(normalize=True))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [5]:
def create_features(df):

    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = ((df["SibSp"] + df["Parch"]) == 0).astype(int)
    df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
    rare_titles = ["Capt", "Col", "Don", "Dr","Jonkheer", "Lady", "Major","Mlle", "Mme", "Ms","Rev", "Sir", "the Countess"]
    df["TitleClean"] = df["Title"].replace(rare_titles,"Rare")
    df["Pclass_Title"] = (df["Pclass"].astype(str) + "_" + df["TitleClean"])
    
    df["AgeBin"] = pd.cut(
    df["Age"],
    bins=[0, 5, 12, 18, 30, 45, 60, 100],
    labels=False)
    
    return df

train = create_features(train)
test = create_features(test)

In [6]:
class GroupAgeImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        self.global_median_ = X["Age"].median()

        self.group_medians_ = (
            X.groupby(
                ["Pclass", "Sex", "TitleClean"]
            )["Age"]
            .median()
        )

        self.ps_medians_ = (
            X.groupby(
                ["Pclass", "Sex"]
            )["Age"]
            .median()
        )

        return self

    def transform(self, X):
        X = X.copy()

        for idx in X.index:
            if pd.isna(X.loc[idx, "Age"]):

                group = (
                    X.loc[idx, "Pclass"],
                    X.loc[idx, "Sex"],
                    X.loc[idx, "TitleClean"]
                )

                if group in self.group_medians_.index:
                    X.loc[idx, "Age"] = self.group_medians_.loc[group]

                else:
                    ps_group = (
                        X.loc[idx, "Pclass"],
                        X.loc[idx, "Sex"]
                    )

                    if ps_group in self.ps_medians_.index:
                        X.loc[idx, "Age"] = self.ps_medians_.loc[ps_group]

                    else:
                        X.loc[idx, "Age"] = self.global_median_

        return X

In [7]:
FEATURES = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "TitleClean",
    "Pclass_Title",
    #"AgeBin"
]

In [8]:
NUMERIC_FEATURES = [
    #"Age",
    "SibSp",
    "Parch",
    "Fare",
]

In [9]:
CATEGORICAL_FEATURES = [
    "Pclass",
    "Sex",
    "Embarked",
    "TitleClean",
    "Pclass_Title",
    #"AgeBin"
]

In [10]:
numeric_pipeline = Pipeline([("imputer",SimpleImputer(strategy="median"))])

categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num",numeric_pipeline,NUMERIC_FEATURES),
                                  ("cat",categorical_pipeline,CATEGORICAL_FEATURES)])

In [11]:
model = LogisticRegression(max_iter=1000)
#model = RandomForestClassifier(n_estimators=300,random_state=42)
fmodel = Pipeline([
    ("age_imputer", GroupAgeImputer()),
    ("preproccessor" , preprocessor), 
    ("classifier", model)])

In [12]:
x = train[FEATURES]
y = train["Survived"]
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
scores = cross_val_score(fmodel,x,y,cv=cv,scoring="accuracy")

In [13]:
print(scores)
print(scores.mean())
print(scores.std())

[0.83240223 0.8258427  0.81460674 0.8258427  0.83146067]
0.8260310087251271
0.006334959971099866


In [14]:
fmodel.fit(x,y)
train_pred = fmodel.predict(x)
train_acc = accuracy_score(y,train_pred)
print(train_acc)

0.8305274971941639


In [15]:
test_x = test[FEATURES]
fmodel.fit(x,y)
test_predictions = fmodel.predict(test_x)

In [16]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

In [17]:
submission.to_csv(
    "submission.csv",
    index=False
)

In [18]:
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         0


In [19]:
pd.crosstab(
    [train["TitleClean"], train["Sex"]],
    train["Survived"],
    normalize="index"
)

Survived                  0         1
TitleClean Sex                       
Master     male    0.425000  0.575000
Miss       female  0.302198  0.697802
Mr         male    0.843327  0.156673
Mrs        female  0.208000  0.792000
Rare       female  0.000000  1.000000
           male    0.750000  0.250000

In [20]:
pd.crosstab(
    [train["Pclass"], train["TitleClean"]],
    train["Survived"],
    normalize="index"
)

Survived                  0         1
Pclass TitleClean                    
1      Master      0.000000  1.000000
       Miss        0.043478  0.956522
       Mr          0.654206  0.345794
       Mrs         0.023810  0.976190
       Rare        0.388889  0.611111
2      Master      0.000000  1.000000
       Miss        0.058824  0.941176
       Mr          0.912088  0.087912
       Mrs         0.097561  0.902439
       Rare        0.888889  0.111111
3      Master      0.607143  0.392857
       Miss        0.500000  0.500000
       Mr          0.887147  0.112853
       Mrs         0.500000  0.500000

In [21]:
train.groupby(
    pd.cut(train["Age"], bins=[0, 5, 12, 18, 30, 45, 60, 100])
)["Survived"].agg(["count", "mean"])

/tmp/ipykernel_16/4288026129.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby(


,count,mean
Age,,
"(0, 5]",44,0.704545
"(5, 12]",25,0.360000
"(12, 18]",70,0.428571
"(18, 30]",270,0.355556
"(30, 45]",202,0.425743
"(45, 60]",81,0.407407
"(60, 100]",22,0.227273


In [22]:
train.groupby(
    ["Pclass", "Sex"]
)["Age"].agg(["count", "mean", "median"])

count       mean  median
Pclass Sex                             
1      female     85  34.611765    35.0
       male      101  41.281386    40.0
2      female     74  28.722973    28.0
       male       99  30.740707    30.0
3      female    102  21.750000    21.5
       male      253  26.507589    25.0

In [23]:
train["Age"].isna().mean()

np.float64(0.19865319865319866)

In [24]:
train.groupby(
    ["Pclass", "Sex", "TitleClean"]
)["Age"].agg(["count", "median"])

count  median
Pclass Sex    TitleClean               
1      female Miss           45    30.0
              Mrs            34    41.5
              Rare            6    28.5
       male   Master          3     4.0
              Mr             87    40.0
              Rare           11    49.0
2      female Miss           32    24.0
              Mrs            41    32.0
              Rare            1    28.0
       male   Master          9     1.0
              Mr             82    31.0
              Rare            8    46.5
3      female Miss           69    18.0
              Mrs            33    31.0
       male   Master         24     4.0
              Mr            229    26.0

In [25]:
train[train["Age"].isna()].groupby(
    ["Pclass", "Sex", "TitleClean"]
).size()

Pclass  Sex     TitleClean
1       female  Miss           1
                Mrs            8
        male    Mr            20
                Rare           1
2       female  Miss           2
        male    Mr             9
3       female  Miss          33
                Mrs            9
        male    Master         4
                Mr            90
dtype: int64

In [26]:
train["Surname"] = train["Name"].str.extract(
    r"^([^,]+)"
)

In [27]:
train.groupby("Surname").size().describe()

count    667.000000
mean       1.335832
std        0.854922
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        9.000000
dtype: float64

In [28]:
train["Surname"].value_counts().head(15)

Surname
Andersson    9
Sage         7
Skoog        6
Panula       6
Carter       6
Goodwin      6
Johnson      6
Rice         5
Fortune      4
Williams     4
Hart         4
Baclini      4
Brown        4
Kelly        4
Ford         4
Name: count, dtype: int64

In [29]:
surname_stats = (
    train.groupby("Surname")
    .agg(
        family_size=("Survived", "size"),
        survival_rate=("Survived", "mean")
    )
)

surname_stats[surname_stats["family_size"] >= 2] \
    .sort_values("family_size", ascending=False) \
    .head(20)

,family_size,survival_rate
Surname,,
Andersson,9,0.222222
Sage,7,0.000000
Johnson,6,0.500000
Carter,6,0.666667
Panula,6,0.000000
Goodwin,6,0.000000
Skoog,6,0.000000
Rice,5,0.000000
Lefebre,4,0.000000


In [30]:
surname_stats[surname_stats["family_size"] >= 2]["survival_rate"].describe()

count    133.000000
mean       0.457811
std        0.388913
min        0.000000
25%        0.000000
50%        0.500000
75%        0.750000
max        1.000000
Name: survival_rate, dtype: float64